# Week 22, Day 2: Virtual environments and pip

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w022d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import sys
print(sys.prefix == sys.base_prefix)

In [ ]:
import sys, os

def in_virtualenv():
    return sys.prefix != sys.base_prefix

def venv_python(venv_folder, windows):
    """Where a venv keeps its python program."""
    if windows:
        return os.path.join(venv_folder, "Scripts", "python.exe")
    return os.path.join(venv_folder, "bin", "python")

print(in_virtualenv())
print(venv_python(".venv", windows=False))

> **Key idea.** One project, one virtual environment: `python3 -m venv .venv`, then activate it. `pip install name` adds packages to the active environment; `pip install -r requirements.txt` installs a project's whole list. Keep `.venv` out of Git and `requirements.txt` in it. Install only packages you trust.

### Exercise: Read requirements.txt

Write `parse_requirements(text)` returning a list of `(name, specifier)` pairs, one for each package line: names are lower-cased, and the specifier is everything after the name (with spaces removed), or `""` if there isn't one. Skip blank lines, and remove comments (everything from `#` on). A name is made of letters, digits, `-`, `_` and `.`; the specifier starts at the first other character.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    text = \"\"\"# What this project needs\nrequests==2.31.0\nRich >= 13.0\npytest~=8.0        # tests only\n\nbeautifulsoup4\nnumpy>=1.24,<2\n\"\"\"\n    assert parse_requirements(text) == [(\"requests\", \"==2.31.0\"), (\"rich\", \">=13.0\"), (\"pytest\", \"~=8.0\"), (\"beautifulsoup4\", \"\"), (\"numpy\", \">=1.24,<2\")]\n    assert parse_requirements(\"\") == []")

### Exercise: Compatible releases

Write `allows(spec, version)` for a single specifier with one of the operators `==`, `>=`, `<=`, `>`, `<`, `!=` or `~=`, and a version like `"2.31.0"` (compare versions as tuples of whole numbers, padding the shorter with zeros). For `~=X.Y` the version must be at least `X.Y` and have the same `X`; for `~=X.Y.Z`, at least `X.Y.Z` with the same `X.Y`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert allows(\"==2.31.0\", \"2.31.0\") and not allows(\"==2.31.0\", \"2.31.1\")\n    assert allows(\">=13.0\", \"13.7.1\") and not allows(\">=13.0\", \"12.9\")\n    assert allows(\"~=2.2\", \"2.9\") and not allows(\"~=2.2\", \"3.0\") and not allows(\"~=2.2\", \"2.1\")\n    assert allows(\"~=2.2.1\", \"2.2.9\") and not allows(\"~=2.2.1\", \"2.3.0\")\n    assert allows(\"!=1.5.0\", \"1.5.1\") and allows(\"<3\", \"2.99\") and allows(\"==2.31\", \"2.31.0\")")

### Exercise: What changed?

Two outputs of `pip freeze` (lines of `name==version`) from before and after some installs. Write `freeze_diff(before, after)` returning a dictionary with sorted lists: `"added"` (names only in after), `"removed"` (names only in before) and `"changed"` (strings like `"rich 13.5.0 -> 13.7.1"` for names in both with different versions). Compare names ignoring case.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    before = \"requests==2.31.0\\nrich==13.5.0\\nurllib3==2.0.4\\n\"\n    after = \"Requests==2.31.0\\nrich==13.7.1\\npytest==8.1.1\\npluggy==1.4.0\\n\"\n    assert freeze_diff(before, after) == {\"added\": [\"pluggy\", \"pytest\"], \"removed\": [\"urllib3\"], \"changed\": [\"rich 13.5.0 -> 13.7.1\"]}\n    assert freeze_diff(\"\", \"\") == {\"added\": [], \"removed\": [], \"changed\": []}")

### Exercise: Install order

`pip` must install a package's dependencies before the package itself. Given `needs` (each package name to the list of packages it depends on), write `install_order(package, needs)` returning the list of packages to install, every dependency before whatever needs it, each once, ending with `package`. Use a depth-first search, and raise `ValueError("cycle")` if packages depend on each other in a loop.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    needs = {\"requests\": [\"urllib3\", \"idna\", \"certifi\", \"charset-normalizer\"], \"urllib3\": [], \"idna\": [], \"certifi\": [], \"charset-normalizer\": [],\n             \"app\": [\"requests\", \"rich\"], \"rich\": [\"markdown-it-py\", \"pygments\"], \"markdown-it-py\": [\"mdurl\"], \"mdurl\": [], \"pygments\": []}\n    order = install_order(\"app\", needs)\n    assert order[-1] == \"app\" and len(order) == len(set(order)) == 10\n    for pkg, deps in needs.items():\n        for d in deps:\n            assert order.index(d) < order.index(pkg)\n    assert install_order(\"idna\", needs) == [\"idna\"]\n    try:\n        install_order(\"a\", {\"a\": [\"b\"], \"b\": [\"a\"]})\n        assert False\n    except ValueError as e:\n        assert str(e) == \"cycle\"")

### Exercise: Typosquatting check

Write `suspicious(name, popular)` returning the popular package name that `name` is suspiciously close to, or `None`: close means not equal but one edit away (one character added, removed or replaced; use the `edit_distance` helper in the starter, from Week 16).

In [ ]:
from functools import cache

def edit_distance(a, b):
    @cache
    def go(i, j):
        if i == len(a):
            return len(b) - j
        if j == len(b):
            return len(a) - i
        if a[i] == b[j]:
            return go(i + 1, j + 1)
        return 1 + min(go(i + 1, j), go(i, j + 1), go(i + 1, j + 1))
    return go(0, 0)

POPULAR = ["requests", "numpy", "pandas", "pytest", "django", "flask", "rich"]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert suspicious(\"reqeusts\", POPULAR) is None\n    assert suspicious(\"request\", POPULAR) == \"requests\" and suspicious(\"numpi\", POPULAR) == \"numpy\"\n    assert suspicious(\"requests\", POPULAR) is None and suspicious(\"matplotlib\", POPULAR) is None\n    assert suspicious(\"djang0\", POPULAR) == \"django\"")

### Bug hunt: A venv in Git

This `.gitignore` check says `.venv/lib/site.py` should be committed. The `.gitignore` text is missing the line that keeps the environment out. Add it, so the program prints `[True, True, False]`.

In [ ]:
GITIGNORE = """__pycache__/
*.log
"""

def ignored(path):
    for rule in GITIGNORE.split():
        if rule.endswith("/") and rule[:-1] in path.split("/")[:-1]:
            return True
        if path.endswith(rule.lstrip("*")):
            return True
    return False

print([ignored(p) for p in [".venv/lib/site.py", "app/__pycache__/x.pyc", "requirements.txt"]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"[True, True, False]\")")

### Bug hunt: Comments in the requirements

This reader treats the comment line and the comment at the end of a line as package names, and prints `['# tools', 'rich', 'pytest  # tests']`. Fix it so it prints `['rich', 'pytest']`.

In [ ]:
TEXT = "# tools\nrich\npytest  # tests\n"
names = [line for line in TEXT.splitlines() if line]
print(names)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"['rich', 'pytest']\")")

### Puzzle: Set up a project

Arrange the commands to set up a new project with its own environment and save its requirements. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # the command to create a venv in the folder .venv (with python3)
b = ___   # True or False: you should commit the .venv folder to Git
c = ___   # does "~=1.4" allow version 1.9? (True or False)
d = ___   # the file that lists a project's packages, by convention

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a.replace(\"  \", \" \") in (\"python3 -m venv .venv\", \"python -m venv .venv\") and b is False and c is True and d == \"requirements.txt\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny resolver

`pip` must pick a version of every package so that all the requirements are satisfied together. Write `resolve(root, index)` where `index` maps each package name to a dictionary from its available versions (strings) to that version's requirements (a list of `(name, spec)` pairs, using single specifiers as in the "compatible releases" exercise, whose `allows` is in the starter). Starting from `root`'s requirements, choose for each package the **newest** version allowed by every requirement placed on it so far, add that version's own requirements, and repeat until nothing changes. Return a dictionary from package name to chosen version, or raise `ValueError(f"no version of {name} fits")` if a package has no version that satisfies all its requirements.

(This simple approach never goes back on a choice; real resolvers backtrack, as in Week 16, when a later requirement rules out an earlier pick.)

In [ ]:
def as_tuple(text, length=None):
    parts = [int(p) for p in text.split(".")]
    if length:
        parts += [0] * (length - len(parts))
    return tuple(parts)

def allows(spec, version):
    for op in ("~=", "==", "!=", ">=", "<=", ">", "<"):
        if spec.startswith(op):
            target = spec[len(op):]
            break
    n = max(len(target.split(".")), len(version.split(".")))
    v, t = as_tuple(version, n), as_tuple(target, n)
    if op == "~=":
        keep = len(target.split(".")) - 1
        return v >= t and v[:keep] == t[:keep]
    return {"==": v == t, "!=": v != t, ">=": v >= t, "<=": v <= t, ">": v > t, "<": v < t}[op]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("INDEX = {\n    \"app\": {\"1.0\": [(\"web\", \">=2.0\"), (\"log\", \"~=1.2\")]},\n    \"web\": {\"1.9\": [(\"log\", \">=1.0\")], \"2.0\": [(\"log\", \">=1.2\"), (\"json5\", \"<3\")], \"2.1\": [(\"log\", \">=1.3\"), (\"json5\", \">=2.5\")]},\n    \"log\": {\"1.1\": [], \"1.2\": [], \"1.3\": [], \"1.4\": [], \"2.0\": []},\n    \"json5\": {\"2.4\": [], \"2.5\": [], \"3.0\": []},\n}\ndef test_values():\n    \"resolves\"\n    assert resolve(\"app\", INDEX) == {\"web\": \"2.1\", \"log\": \"1.4\", \"json5\": \"3.0\"}\n    small = {\"a\": {\"1\": [(\"b\", \"<2\")]}, \"b\": {\"1.5\": [], \"2.0\": []}}\n    assert resolve(\"a\", small) == {\"b\": \"1.5\"}\ndef test_conflict():\n    \"no fit\"\n    bad = {\"a\": {\"1\": [(\"b\", \">=2\"), (\"c\", \">=1\")]}, \"b\": {\"2.0\": [(\"c\", \"<1\")]}, \"c\": {\"0.5\": [], \"1.0\": []}}\n    try:\n        resolve(\"a\", bad)\n        assert False, \"should fail\"\n    except ValueError as e:\n        assert str(e) == \"no version of c fits\"")

## Warm-up

`sys.prefix` is where the **running** Python keeps its packages, and `sys.base_prefix` is the Python it was made from. They're the same unless the program is running inside a virtual environment. By the end of today, on your own computer, this will print `False`, and that's what you want.

## Why environments

Picture two projects on one computer:

- your website project needs version 2 of a package;
- a course exercise needs version 3, which changed how it works.

With one shared Python, installing version 3 for the exercise breaks the website. And when you share a project, the other person has no idea which packages, at which versions, it needs.

A **virtual environment** ("venv") is a folder holding its own copy of the Python program's launcher and its own `site-packages` folder for packages. Activating it puts its Python first on `PATH`, so `python` and `pip` mean **this project's** Python and `pip`. Every project gets its own, usually in a folder called `.venv` inside the project.

- Packages installed in one environment are invisible to every other.
- Deleting the `.venv` folder removes the environment completely; nothing else is affected.
- You never put `.venv` in Git (it's big, and specific to your computer); you record **what's in it** instead, in `requirements.txt`.

## Making one

In a terminal, in your project folder:

```
python3 -m venv .venv                 (Windows: py -m venv .venv)
```

That makes the `.venv` folder. To use it, **activate** it:

```
source .venv/bin/activate             (macOS and Linux)
.venv\Scripts\activate                (Windows PowerShell)
```

The prompt changes to show `(.venv)` at the start. Now:

```
(.venv) $ which python                (or: where.exe python on Windows)
/home/you/atlas/.venv/bin/python
(.venv) $ python -c "import sys; print(sys.prefix)"
/home/you/atlas/.venv
```

Inside an activated environment, `python` and `pip` always mean the environment's own, on every system. `deactivate` turns it off. Activation only lasts for that terminal window: activate again in each new terminal (code editors such as VS Code do it for you; Day 3).

## pip

**PyPI**, the Python Package Index (pypi.org), is where packages are published. `pip` downloads them from there and installs them into the active environment.

| Command | Does |
|---|---|
| `pip install requests` | install the newest version (and whatever it needs) |
| `pip install "requests==2.31.0"` | install an exact version |
| `pip install --upgrade requests` | upgrade to the newest |
| `pip uninstall requests` | remove it |
| `pip list` | list installed packages and versions |
| `pip show requests` | details: version, summary, what it needs |
| `pip freeze` | every installed package as `name==version`, for saving |
| `pip install -r requirements.txt` | install everything listed in a file |

A few things to know:

- The **package name** you install isn't always the **module name** you import: `pip install beautifulsoup4` gives `import bs4`, and `pip install pillow` gives `import PIL`. The package's page on PyPI says.
- Packages depend on other packages. `pip install requests` also installs `urllib3`, `idna`, `certifi` and `charset-normalizer`. `pip` works out these **dependencies** for you.
- Only install packages you trust. Anyone can publish to PyPI, and fake packages with names one letter away from popular ones ("typosquatting") have been used to spread malware. Check the spelling, the download count and the project's homepage.
- If `pip install` says "externally managed environment", you're not in a venv: create and activate one.

(In this website's browser Python, many popular pure-Python packages can be loaded with `micropip`, but the habits above are for your own computer.)

## requirements.txt

To let someone (or your future self, on a new computer) rebuild your environment, list the packages in a file called `requirements.txt` in the project folder:

```
# What this project needs
requests==2.31.0
rich>=13.0
pytest~=8.0        # tests only
```

Then anyone can do:

```
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

**Version specifiers** say which versions are acceptable:

| Specifier | Means |
|---|---|
| `==2.31.0` | exactly this version |
| `>=13.0` | this or newer |
| `<3` | older than 3 |
| `>=1.4,<2` | both conditions |
| `~=2.2` | "compatible release": at least 2.2, but below 3.0 |
| `~=2.2.1` | at least 2.2.1, but below 2.3 |
| `!=1.5.0` | anything but this |

`pip freeze > requirements.txt` writes every installed package with its exact version, which gives an exactly repeatable environment; hand-written files with ranges are easier to keep up to date. Bigger projects describe their requirements in a `pyproject.toml` file instead, and newer tools such as `uv` manage environments and requirements together; the ideas are the same.

## Good habits

1. Make the venv first, before installing anything for a project.
2. Add `.venv/` to `.gitignore` straight away.
3. Keep `requirements.txt` up to date whenever you add a package, and commit it.
4. If an environment gets into a mess, delete `.venv` and rebuild it from `requirements.txt`: that's what the file is for.
5. "Module not found" right after installing? You're probably running a different Python from the one `pip` installed into: activate the venv, then check `python -c "import sys; print(sys.executable)"` and `pip --version`.

## Practice

## Quiz

## Daily challenge

## Before you go

In your `atlas` folder on your computer: make a virtual environment, activate it, `pip install rich`, and run `python -c "from rich import print; print('[bold green]It works![/]')"`. Save `requirements.txt` with `pip freeze`, add `.venv/` to `.gitignore`, and commit. Tomorrow: VS Code and its debugger, which shows your program running line by line, like this site's X-ray.